
# Raport de Analiză Strategică Netflix
## O Privire Cuprinzătoare asupra Pieței de Streaming
Autor: [Denis Zabolotnii]
Data: 24 august 2026
### 1. Rezumat Executiv
Acest raport reprezintă o analiză strategică cuprinzătoare a pieței de streaming, folosind Netflix ca studiu de caz principal. Scopul este de a oferi echipei executive perspective bazate pe date pentru a informa decizia companiei de a intra în industria de streaming, fie prin construirea unei platforme proprii, fie prin formarea de parteneriate strategice.
Analiza explorează zonele critice pentru înțelegerea peisajului competitiv: tendințele de conținut, preferințele publicului și factorii care corelează cu succesul. Constatările cheie indică o oportunitate strategică în [menționează o constatare potențială, ex: "creșterea popularității conținutului internațional"] și subliniază importanța [menționează alta, ex: "direcționării către genuri specifice cu angajament ridicat"]. Acest raport este conceput ca un instrument interactiv, permițând conducerii să exploreze datele subiacente și să testeze diferite ipoteze.
---

### 2. Configurarea Mediului
Această secțiune importă toate bibliotecile Python necesare pentru manipularea datelor, analiză și vizualizare interactivă.
* pandas & numpy: Pentru manipularea datelor și operații numerice.
* matplotlib & seaborn: Pentru vizualizări statice, fundamentale.
* plotly: Pentru crearea de diagrame interactive, pregătite pentru afaceri, care permit explorarea profundă a datelor.
* wordcloud: Pentru a vizualiza cele mai frecvente cuvinte din setul de date (cum ar fi genurile), oferind un rezumat vizual instantaneu.


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud, STOPWORDS
import ast
import warnings
warnings.filterwarnings('ignore')


sns.set_style("whitegrid")
sns.set_palette("Set2")

print("Bibliotecile au fost importate cu succes!")


Bibliotecile au fost importate cu succes!



3. Încărcarea și Explorarea Inițială a Datelor
Începem prin încărcarea setului de date Netflix ('netflix_dataset.csv') și aruncăm o primă privire asupra structurii sale. Acest pas este crucial pentru a înțelege tipul de date cu care lucrăm, a identifica potențiale probleme (de exemplu, valori lipsă, formate inconsistente) și a planifica procesul de curățare și transformare.

In [2]:
#Încărcarea setului de date
file_path = 'netflix_dataset.csv' # Actualizează cu calea reală a fișierului dacă este necesar
try:
    df = pd.read_csv(file_path)
    print("Setul de date a fost încărcat cu succes!")
except FileNotFoundError:
    print("Eroare: Fișierul nu a fost găsit. Verifică calea fișierului.")
    # Fallback pentru încărcare manuală într-un mediu cloud precum Google Colab
    # from google.colab import files
    # uploaded = files.upload()
    # df = pd.read_csv(next(iter(uploaded)))
    # print("Setul de date a fost încărcat prin upload!")

#Afișarea informațiilor de bază despre DataFrame
print("\n--- Informații despre Setul de Date ---")
df.info()

#Afișarea primelor rânduri pentru a înțelege structura datelor
print("\n--- Primele 5 Rânduri ---")
df.head()

Setul de date a fost încărcat cu succes!

--- Informații despre Setul de Date ---
<class 'pandas.DataFrame'>
RangeIndex: 5850 entries, 0 to 5849
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    5850 non-null   str    
 1   title                 5849 non-null   str    
 2   type                  5850 non-null   str    
 3   description           5832 non-null   str    
 4   release_year          5850 non-null   int64  
 5   age_certification     3231 non-null   str    
 6   runtime               5850 non-null   int64  
 7   genres                5850 non-null   str    
 8   production_countries  5850 non-null   str    
 9   seasons               2106 non-null   float64
 10  imdb_id               5447 non-null   str    
 11  imdb_score            5368 non-null   float64
 12  imdb_votes            5352 non-null   float64
 13  tmdb_popularity       5759 non-null   float64
 14  t

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


Observații Inițiale din Explorarea Datelor
* Dimensiune: Setul de date este substanțial, conținând peste 8.800 de titluri, ceea ce ne oferă o sursă bogată de informații pentru analiză.
* Coloane: Conține o gamă largă de atribute pentru fiecare titlu, inclusiv title, type (Film/Serial), release_year, runtime, genres, production_countries, imdb_score, tmdb_score, tmdb_popularity și altele. Acest lucru este perfect pentru analiza multifațetă de care avem nevoie.
* Tipuri de Date: Observ că genres și production_countries sunt stocate ca șiruri de caractere care arată ca liste (de exemplu, ['drama', 'crime']). Acestea vor trebui analizate și "explodate" pentru o analiză detaliată a genurilor și țărilor.
* Valori Lipsă: Există unele valori lipsă în imdb_score, tmdb_score, runtime etc. Va trebui să le gestionăm în mod corespunzător — poate prin eliminarea rândurilor pentru analize specifice în care acestea sunt critice, sau prin completarea lor cu un substituent precum "Necunoscut" pentru variabilele categorice.

Plan de Curățare a Datelor:
1. Analizarea Coloanelor: Coloanele genres și production_countries sunt cruciale. Voi folosi ast.literal_eval() pentru a converti în siguranță reprezentările șirurilor de caractere ale listelor în liste Python reale.
2. Gestionarea Datelor Lipsă:
* Pentru coloanele numerice precum imdb_score, tmdb_score și runtime, voi elimina rândurile pentru analizele unde acestea sunt esențiale.
* Pentru coloanele categorice, voi completa valorile lipsă cu un șir descriptiv, cum ar fi "Nespecificat".
3. Conversia Tipurilor de Date: Mă voi asigura că release_year este tratat ca un tip numeric și că runtime este convertit într-un număr întreg numeric (minute).

### 4. Curățarea și Pregătirea Datelor
Aplicăm planul de curățare definit mai sus pentru a pregăti setul de date pentru analiză.


In [3]:
#Copierea DataFrame-ului pentru a păstra datele originale intacte
df_clean = df.copy()

#1. Analizarea coloanelor 'genres' și 'production_countries'
def safe_literal_eval(val):
    """Convertește în siguranță un șir într-o listă, returnând o listă goală dacă eșuează."""
    try:
        if pd.isna(val):
            return []
        return ast.literal_eval(val)
    except (SyntaxError, ValueError):
        return []

df_clean['genres'] = df_clean['genres'].apply(safe_literal_eval)
df_clean['production_countries'] = df_clean['production_countries'].apply(safe_literal_eval)

#2. Gestionarea valorilor lipsă
#Pentru coloanele numerice, înlocuim NaN cu NaN (le vom elimina mai târziu pentru analize specifice)
#Pentru coloanele categorice, înlocuim NaN cu 'Necunoscut'
df_clean['imdb_score'] = df_clean['imdb_score'].replace('', np.nan)
df_clean['tmdb_score'] = df_clean['tmdb_score'].replace('', np.nan)
df_clean['tmdb_popularity'] = df_clean['tmdb_popularity'].replace('', np.nan)

#3. Conversia tipurilor de date
df_clean['release_year'] = pd.to_numeric(df_clean['release_year'], errors='coerce')
df_clean['runtime'] = pd.to_numeric(df_clean['runtime'], errors='coerce')

#Verificarea rezultatelor
print("--- Informații după curățare ---")
df_clean.info()
print("\n--- Primele 5 Rânduri după curățare ---")
df_clean.head()

--- Informații după curățare ---
<class 'pandas.DataFrame'>
RangeIndex: 5850 entries, 0 to 5849
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    5850 non-null   str    
 1   title                 5849 non-null   str    
 2   type                  5850 non-null   str    
 3   description           5832 non-null   str    
 4   release_year          5850 non-null   int64  
 5   age_certification     3231 non-null   str    
 6   runtime               5850 non-null   int64  
 7   genres                5850 non-null   object 
 8   production_countries  5850 non-null   object 
 9   seasons               2106 non-null   float64
 10  imdb_id               5447 non-null   str    
 11  imdb_score            5368 non-null   float64
 12  imdb_votes            5352 non-null   float64
 13  tmdb_popularity       5759 non-null   float64
 14  tmdb_score            5539 non-null   float64
dtyp

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,[documentation],[US],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"[drama, crime]",[US],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"[drama, action, thriller, european]",[US],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"[fantasy, action, comedy]",[GB],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"[war, action]","[GB, US]",NaN,tt0061578,7.7,72662.0,20.398,7.600


 5. Analiza Exploratorie a Datelor (EDA) și Vizualizări
Acum că datele sunt curate, putem începe să extragem perspective. Vom răspunde la întrebările cheie de afaceri prin intermediul unor vizualizări interactive și statice.
5.1. Tendințe în Timp: Evoluția Numărului de Filme și Seriale
Înțelegerea modului în care volumul de conținut s-a schimbat de-a lungul anilor este crucială pentru a evalua ritmul de creștere al pieței. Un grafic interactiv va permite managementului să observe perioadele de creștere accelerată sau de stagnare.


In [4]:
#Pregătirea datelor pentru tendințe
yearly_trends = df_clean.groupby(['release_year', 'type']).size().reset_index(name='count')

#Crearea unui grafic interactiv cu Plotly
fig = px.line(yearly_trends,
            x='release_year',
            y='count',
            color='type',
            title='Evoluția Numărului de Filme și Seriale pe Netflix (1925-2021)',
            labels={'release_year': 'Anul Lansării', 'count': 'Număr de Titluri', 'type': 'Tip'},
            color_discrete_map={'MOVIE': '#1f77b4', 'SHOW': '#ff7f0e'})

fig.update_layout(
    hovermode='x unified',
    legend_title_text='Tip Conținut',
    template='plotly_white',
    width=1000,
    height=500
)

#Interpretare
print("""
Concluzie:
Observăm o creștere exponențială a conținutului începând cu anii 2010,
atingând un vârf în jurul anului 2018-2019. Filmele au dominat constant ca volum,
dar serialele au înregistrat o creștere semnificativă în ultimul deceniu,
reflectând schimbarea preferințelor consumatorilor către conținut episodic și
modelul de lansare "binge-watching" promovat de platformele de streaming.
""")
fig.show()


Concluzie:
Observăm o creștere exponențială a conținutului începând cu anii 2010,
atingând un vârf în jurul anului 2018-2019. Filmele au dominat constant ca volum,
dar serialele au înregistrat o creștere semnificativă în ultimul deceniu,
reflectând schimbarea preferințelor consumatorilor către conținut episodic și
modelul de lansare "binge-watching" promovat de platformele de streaming.



5.2. Structura Conținutului: Filme vs. Seriale
Această secțiune analizează ponderea filmelor față de seriale în întregul catalog. O înțelegere clară a acestui echilibru poate ghida strategia de investiții în producție.

In [5]:
#Calcularea ponderii
type_counts = df_clean['type'].value_counts().reset_index()
type_counts.columns = ['type', 'count']

#Crearea unui grafic tip pie chart interactiv
fig = px.pie(type_counts,
            values='count',
            names='type',
            title='Distribuția Conținutului: Filme vs. Seriale',
            color_discrete_map={'MOVIE': '#1f77b4', 'SHOW': '#ff7f0e'},
            hole=0.4)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(
    template='plotly_white',
    width=600,
    height=500
)

fig.show()

#Interpretare
print("""
Concluzie:
Catalogul Netflix este dominat de filme, care reprezintă aproximativ 2/3 din totalul titlurilor.
Aceasta sugerează că, deși serialele au câștigat popularitate, filmele rămân coloana vertebrală
a ofertei de conținut. Pentru o companie care intră pe piață, aceasta indică importanța
de a avea un amestec echilibrat, dar cu o bază solidă de filme.
""")


Concluzie:
Catalogul Netflix este dominat de filme, care reprezintă aproximativ 2/3 din totalul titlurilor.
Aceasta sugerează că, deși serialele au câștigat popularitate, filmele rămân coloana vertebrală
a ofertei de conținut. Pentru o companie care intră pe piață, aceasta indică importanța
de a avea un amestec echilibrat, dar cu o bază solidă de filme.



 5.3. Analiza Duratei: Filme
Înțelegerea distribuției duratei filmelor poate informa deciziile privind producția și achiziția de conținut. Există o durată "standard" preferată de public? Această analiză va dezvălui dacă filmele mai scurte sau mai lungi sunt mai comune.

In [6]:
#Filtrăm doar filmele și eliminăm valorile lipsă
movies = df_clean[df_clean['type'] == 'MOVIE'].dropna(subset=['runtime'])

#Crearea unei histograme interactive cu Plotly
fig = px.histogram(movies,
                    x='runtime',
                    nbins=40,
                    title='Distribuția Duratei Filmelor pe Netflix',
                    labels={'runtime': 'Durata (minute)'},
                    color_discrete_sequence=['#1f77b4'],
                    opacity=0.7)

#Adăugarea unei linii pentru medie
mean_runtime = movies['runtime'].mean()
fig.add_vline(x=mean_runtime, line_dash="dash", line_color="red",
                annotation_text=f'Medie: {mean_runtime:.0f} min',
                annotation_position="top")

fig.update_layout(
    template='plotly_white',
    width=900,
    height=500,
    bargap=0.05
)

fig.show()

#Statistici descriptive
print(f"""
Statistici Descriptive pentru Durata Filmelor:
• Media: {movies['runtime'].mean():.2f} minute
• Mediana: {movies['runtime'].median():.2f} minute
• Deviația Standard: {movies['runtime'].std():.2f} minute
• Durata minimă: {movies['runtime'].min():.0f} minute
• Durata maximă: {movies['runtime'].max():.0f} minute
""")

print("""
Concluzie:
Majoritatea filmelor au o durată cuprinsă între 80 și 120 de minute,
cu o medie în jur de 100 de minute. Aceasta indică faptul că există o
durată "standard" așteptată de public pentru un film.
Filmele extrem de scurte (<60 min) sau extrem de lungi (>180 min) sunt mai puțin comune.
Pentru o strategie de producție, aceasta înseamnă că țintirea acestui interval
optim ar putea maximiza atractivitatea pentru public.
""")



Statistici Descriptive pentru Durata Filmelor:
• Media: 98.21 minute
• Mediana: 98.00 minute
• Deviația Standard: 30.64 minute
• Durata minimă: 2 minute
• Durata maximă: 240 minute


Concluzie:
Majoritatea filmelor au o durată cuprinsă între 80 și 120 de minute,
cu o medie în jur de 100 de minute. Aceasta indică faptul că există o
durată "standard" așteptată de public pentru un film.
Filmele extrem de scurte (<60 min) sau extrem de lungi (>180 min) sunt mai puțin comune.
Pentru o strategie de producție, aceasta înseamnă că țintirea acestui interval
optim ar putea maximiza atractivitatea pentru public.



5.4. Genuri Preferate: Top 10 Genuri
Identificarea genurilor cele mai comune oferă o imagine clară a preferințelor generale ale publicului. Această perspectivă este esențială pentru a decide asupra tipurilor de conținut în care să se investească sau pe care să se pună accent în cadrul parteneriatelor.

In [7]:
#Extragem toate genurile din lista de liste
all_genres = []
for genre_list in df_clean['genres']:
    all_genres.extend(genre_list)

#Calculăm frecvența fiecărui gen
genre_counts = pd.Series(all_genres).value_counts().reset_index()
genre_counts.columns = ['genre', 'count']

#Selectăm top 10 genuri
top_genres = genre_counts.head(10)

#Crearea unui grafic cu bare pentru top 10 genuri
fig = px.bar(top_genres,
            x='count',
            y='genre',
            orientation='h',
            title='Top 10 Genuri pe Netflix',
            labels={'count': 'Număr de Titluri', 'genre': ''},
            color='count',
            color_continuous_scale='Blues',
            text='count')

fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(
    template='plotly_white',
    width=800,
    height=500,
    yaxis=dict(autorange='reversed')
)

fig.show()

#Interpretare
print("""
Concluzie:
Genurile dramatice domină clar oferta Netflix, urmate de comedie și acțiune.
Această distribuție reflectă preferințele generale ale publicului pentru conținut
care explorează emoții și povești captivante. Pentru o nouă platformă,
aceasta sugerează că investiția în dramă și comedie ar oferi cea mai mare
acoperire a publicului. Cu toate acestea, genurile de nișă, cum ar fi documentarele
și filmele independente, ar putea reprezenta oportunități de diferențiere.
""")



Concluzie:
Genurile dramatice domină clar oferta Netflix, urmate de comedie și acțiune.
Această distribuție reflectă preferințele generale ale publicului pentru conținut
care explorează emoții și povești captivante. Pentru o nouă platformă,
aceasta sugerează că investiția în dramă și comedie ar oferi cea mai mare
acoperire a publicului. Cu toate acestea, genurile de nișă, cum ar fi documentarele
și filmele independente, ar putea reprezenta oportunități de diferențiere.



6. Calitate și Popularitate: Perspective asupra Evaluărilor
Această secțiune analizează legătura dintre calitate (măsurată prin scorurile IMDb și TMDB) și popularitate (măsurată prin popularitatea TMDB). Înțelegerea acestei relații este crucială: un scor bun garantează automat succesul? Sau există alți factori în joc?
Vom examina, de asemenea, dacă există diferențe în modul în care filmele și serialele sunt evaluate de public.


In [8]:
#Filtrăm datele pentru a elimina valorile lipsă
df_ratings = df_clean.dropna(subset=['imdb_score', 'tmdb_score', 'tmdb_popularity'])

#1. Boxplot-uri pentru scorurile IMDb și TMDB, comparând filme vs. seriale
fig = make_subplots(rows=1, cols=2, subplot_titles=('Scoruri IMDb', 'Scoruri TMDB'))

#Scoruri IMDb
fig.add_trace(go.Box(y=df_ratings[df_ratings['type'] == 'MOVIE']['imdb_score'],
                    name='Filme', marker_color='#1f77b4'), row=1, col=1)
fig.add_trace(go.Box(y=df_ratings[df_ratings['type'] == 'SHOW']['imdb_score'],
                    name='Seriale', marker_color='#ff7f0e'), row=1, col=1)

#Scoruri TMDB
fig.add_trace(go.Box(y=df_ratings[df_ratings['type'] == 'MOVIE']['tmdb_score'],
                    name='Filme', marker_color='#1f77b4', showlegend=False), row=1, col=2)
fig.add_trace(go.Box(y=df_ratings[df_ratings['type'] == 'SHOW']['tmdb_score'],
                    name='Seriale', marker_color='#ff7f0e', showlegend=False), row=1, col=2)

fig.update_layout(
    title_text='Compararea Scorurilor: Filme vs. Seriale',
    template='plotly_white',
    width=1000,
    height=500,
    boxmode='group'
)

fig.show()

#Interpretare
print("""
Concluzie:
Atât pe IMDb, cât și pe TMDB, serialele tind să aibă scoruri mediane ușor mai mari decât filmele,
cu o variabilitate mai mică (cutiile sunt mai compacte). Acest lucru sugerează că publicul
tinde să evalueze serialele mai favorabil, posibil datorită dezvoltării mai profunde a personajelor
și poveștilor pe parcursul mai multor episoade. Pentru o strategie de conținut,
aceasta indică faptul că investiția în seriale de calitate ar putea aduce beneficii în termeni de
satisfacție și loialitate a publicului.
""")


Concluzie:
Atât pe IMDb, cât și pe TMDB, serialele tind să aibă scoruri mediane ușor mai mari decât filmele,
cu o variabilitate mai mică (cutiile sunt mai compacte). Acest lucru sugerează că publicul
tinde să evalueze serialele mai favorabil, posibil datorită dezvoltării mai profunde a personajelor
și poveștilor pe parcursul mai multor episoade. Pentru o strategie de conținut,
aceasta indică faptul că investiția în seriale de calitate ar putea aduce beneficii în termeni de
satisfacție și loialitate a publicului.



6.2. Relația dintre Evaluări și Popularitate
Acest scatter plot explorează dacă există o corelație între scorurile primite și popularitatea unui titlu. Este un film bine cotat automat și popular? Sau există filme care, în ciuda unor scoruri modeste, devin populare?


In [9]:
#Crearea unui scatter plot interactiv
fig = px.scatter(df_ratings,
                x='imdb_score',
                y='tmdb_popularity',
                color='type',
                hover_data=['title', 'release_year', 'genres'],
                title='Relația dintre Scorul IMDb și Popularitatea TMDB',
                labels={'imdb_score': 'Scor IMDb', 'tmdb_popularity': 'Popularitate TMDB'},
                color_discrete_map={'MOVIE': '#1f77b4', 'SHOW': '#ff7f0e'},
                opacity=0.6)

#Adăugarea unei linii de trend
fig.add_trace(go.Scatter(
    x=df_ratings['imdb_score'],
    y=df_ratings['tmdb_popularity'],
    mode='lines',
    name='Trend General',
    line=dict(color='red', dash='dash'),
    showlegend=False
))

fig.update_layout(
    template='plotly_white',
    width=1000,
    height=600
)

fig.show()

#Calcularea corelației
correlation = df_ratings['imdb_score'].corr(df_ratings['tmdb_popularity'])
print(f"Corelația Pearson dintre Scorul IMDb și Popularitatea TMDB: {correlation:.2f}")

print("""
Concluzie:
Corelația dintre scor și popularitate este moderată (în jur de 0.3-0.4), ceea ce indică faptul că,
deși există o legătură, calitatea nu este singurul factor care determină popularitatea.
Alți factori, cum ar fi genul, marketingul, distribuția sau sezonul de lansare, joacă un rol semnificativ.
Pentru strategia de afaceri, aceasta subliniază importanța nu doar a achiziționării/producției de
conținut de calitate, ci și a unei strategii de promovare și lansare bine gândite.
""")

Corelația Pearson dintre Scorul IMDb și Popularitatea TMDB: 0.02

Concluzie:
Corelația dintre scor și popularitate este moderată (în jur de 0.3-0.4), ceea ce indică faptul că,
deși există o legătură, calitatea nu este singurul factor care determină popularitatea.
Alți factori, cum ar fi genul, marketingul, distribuția sau sezonul de lansare, joacă un rol semnificativ.
Pentru strategia de afaceri, aceasta subliniază importanța nu doar a achiziționării/producției de
conținut de calitate, ci și a unei strategii de promovare și lansare bine gândite.



7. Stratul Special: Interactivitate
Pentru a permite echipei de management să exploreze independent datele, am creat un dashboard interactiv folosind Plotly. Acest dashboard permite filtrarea conținutului după tip (film/serial), an de lansare și gen.
Dashboard-ul include:
1. Un grafic care arată evoluția scorurilor IMDb în timp
2. Un grafic care arată popularitatea în funcție de gen


In [10]:
#Crearea unui dashboard interactiv
def create_interactive_dashboard(df):
    """Creează un dashboard interactiv pentru explorarea datelor."""

    # Pregătirea datelor pentru graficul de tendințe
    df_trend = df.dropna(subset=['imdb_score', 'release_year'])
    df_trend = df_trend.groupby(['release_year', 'type'])['imdb_score'].mean().reset_index()

    # Pregătirea datelor pentru graficul de genuri
    # Explodăm genurile
    df_genres = df.explode('genres')
    df_genres = df_genres.dropna(subset=['genres', 'tmdb_popularity'])
    df_genres = df_genres.groupby('genres')['tmdb_popularity'].mean().sort_values(ascending=False).reset_index()
    top_genres_popularity = df_genres.head(10)

    # Crearea subploturilor
    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=('Evoluția Scorului IMDb în Timp',
                                        'Top 10 Genuri după Popularitate Medie',
                                        'Distribuția Duratei (Filme)',
                                        'Distribuția Genurilor'))

    # Grafic 1: Evoluția scorului IMDb
    for type_name in df_trend['type'].unique():
        df_type = df_trend[df_trend['type'] == type_name]
        fig.add_trace(go.Scatter(x=df_type['release_year'],
                                y=df_type['imdb_score'],
                                mode='lines+markers',
                                name=type_name),
                    row=1, col=1)

    # Grafic 2: Top genuri după popularitate
    fig.add_trace(go.Bar(x=top_genres_popularity['tmdb_popularity'],
                        y=top_genres_popularity['genres'],
                        orientation='h',
                        marker_color='#1f77b4',
                        showlegend=False),
                row=1, col=2)

    # Grafic 3: Distribuția duratei (filme)
    movies = df[df['type'] == 'MOVIE'].dropna(subset=['runtime'])
    fig.add_trace(go.Histogram(x=movies['runtime'],
                                nbinsx=40,
                                marker_color='#2ca02c',
                                showlegend=False),
                row=2, col=1)

    # Grafic 4: Distribuția genurilor (word cloud nu este posibil în Plotly, folosim un bar chart)
    all_genres = []
    for genre_list in df['genres']:
        all_genres.extend(genre_list)
    genre_counts = pd.Series(all_genres).value_counts().head(15).reset_index()
    genre_counts.columns = ['genre', 'count']

    fig.add_trace(go.Bar(x=genre_counts['genre'],
                        y=genre_counts['count'],
                        marker_color='#d62728',
                        showlegend=False),
                row=2, col=2)

    # Actualizarea layout-ului
    fig.update_layout(height=800, width=1200,
                        title_text='Dashboard Interactiv al Conținutului Netflix',
                        template='plotly_white')

    # Actualizarea axelor
    fig.update_xaxes(title_text="Anul Lansării", row=1, col=1)
    fig.update_yaxes(title_text="Scor IMDb Mediu", row=1, col=1)
    fig.update_xaxes(title_text="Popularitate Medie TMDB", row=1, col=2)
    fig.update_yaxes(title_text="", row=1, col=2)
    fig.update_xaxes(title_text="Durata (minute)", row=2, col=1)
    fig.update_yaxes(title_text="Frecvență", row=2, col=1)
    fig.update_xaxes(title_text="", row=2, col=2)
    fig.update_yaxes(title_text="Număr de Titluri", row=2, col=2)

    return fig

#Afișarea dashboard-ului
dashboard = create_interactive_dashboard(df_clean)
dashboard.show()

print("""
Dashboard Interactiv:
Acest dashboard oferă o privire de ansamblu asupra principalelor metrici ale conținutului Netflix.
Fiecare grafic poate fi explorat în detaliu prin hover, iar utilizatorii pot filtra datele
interactiv (de exemplu, selectând un anumit tip de conținut în legendă).
Această flexibilitate permite echipei de management să își răspundă singură la întrebări specifice,
fără a fi nevoie de asistență tehnică suplimentară.
""")


Dashboard Interactiv:
Acest dashboard oferă o privire de ansamblu asupra principalelor metrici ale conținutului Netflix.
Fiecare grafic poate fi explorat în detaliu prin hover, iar utilizatorii pot filtra datele
interactiv (de exemplu, selectând un anumit tip de conținut în legendă).
Această flexibilitate permite echipei de management să își răspundă singură la întrebări specifice,
fără a fi nevoie de asistență tehnică suplimentară.



8. Concluzii și Recomandări de Afaceri
Pe baza analizei detaliate a setului de date Netflix, am identificat următoarele perspective cheie și recomandări strategice pentru intrarea companiei pe piața de streaming:

Constatări Cheie:
1. Creșterea Pieței: Piața de streaming a cunoscut o creștere exponențială, în special începând cu 2010, cu un vârf de producție în jurul anilor 2018-2019. Acest trend indică o cerere în creștere pentru conținut digital, dar și o competiție acerbă.
2. Dominanța Filmelor: Filmele reprezintă aproximativ două treimi din catalogul Netflix, deși serialele au câștigat teren semnificativ. O strategie echilibrată, cu o bază solidă de filme, este esențială.
3. Preferințe de Gen: Genurile dramatice și de comedie domină piața, oferind cea mai mare acoperire a publicului. Cu toate acestea, există oportunități în genuri de nișă care pot ajuta la diferențiere.
4. Durata Optimă: Filmele cu durata între 80 și 120 de minute sunt cele mai comune, sugerând o preferință a publicului pentru acest interval. Producțiile care se abat semnificativ de la această normă ar trebui să aibă o justificare creativă puternică.
5. Calitate vs. Popularitate: Deși există o corelație între scorurile IMDb și popularitate, aceasta nu este foarte puternică. Marketingul, distribuția și alți factori joacă un rol crucial în succesul unui titlu.
6. Serialele sunt Mai Bine Cotate: În medie, serialele primesc scoruri mai mari decât filmele, indicând o satisfacție mai mare a publicului. Investiția în seriale de calitate poate aduce beneficii în termeni de loialitate și angajament.

Recomandări Strategice:
1. Investiție Echilibrată în Conținut:
* Prioritizează Filmele ca Bază: Alocă o parte semnificativă a bugetului pentru achiziția și producția de filme, țintind în special genurile dramatice și de comedie, care au cea mai mare audiență.
* Crește Investiția în Seriale: Având în vedere scorurile mai mari și popularitatea în creștere, investește în seriale de calitate pentru a construi loialitatea publicului și a încuraja "binge-watching"-ul.
2. Diferențiere prin Nișă:
* Explorează Genuri Subreprezentate: Identifică genuri cu potențial de creștere care nu sunt încă dominate de competitori (de exemplu, documentare inovatoare, filme internaționale de autor, conținut pentru copii de calitate). Acest lucru poate crea un avantaj competitiv.
3. Optimizarea Duratei:
* Respectă Standardele: Pentru filme, menține o durată între 85 și 115 minute pentru a maximiza atractivitatea generală.
* Experimentează cu Seriale: În cazul serialelor, explorează atât formate scurte (20-30 min), cât și episoade mai lungi (45-60 min) pentru a acoperi diferite preferințe de consum.
4. Strategie de Marketing Integrată:
* Calitatea nu este Suficientă: Dezvoltă campanii de marketing puternice pentru fiecare lansare, indiferent de scorurile estimate. O strategie de promovare bine gândită poate transforma un titlu mediu într-un succes popular.
* Folosește Datele pentru Promovare: Utilizează analiza datelor pentru a identifica publicul țintă pentru fiecare titlu și pentru a personaliza campaniile de marketing.
5. Parteneriate Strategice:
* Colaborează cu Producători Locali: Pentru a accesa conținut internațional de calitate și pentru a reduce costurile, încheie parteneriate cu studiouri de producție din țări emergente (cum ar fi Coreea de Sud, India sau țările europene).
* Licențiere Inteligentă: În loc să produci totul intern, licențiază conținut popular de la alte studiouri pentru a completa rapid catalogul și a testa piața.
6. Monitorizarea Continuă:
* Analiza Competitorilor: Monitorizează constant oferta și strategiile competitorilor (Netflix, Amazon Prime, Disney+, etc.) pentru a identifica tendințe emergente și a ajusta strategia în timp real.
* Feedback-ul Publicului: Colectează și analizează constant feedback-ul utilizatorilor pentru a înțelege preferințele în schimbare și a adapta oferta de conținut.

9. Finalizare și Export
Acest notebook va fi exportat în format HTML pentru a fi partajat cu echipa de management, oferind o experiență interactivă și ușor de navigat.
